# vLLM Speedup Benchmark (HF generate vs vLLM)

Produces a first-party speedup number for whitepaper §6.4 — the section currently says "3× to 20× depending on configuration" with no measurement of its own. This notebook fixes that by running the exact configuration §6.4 names as the validation target, plus a sweep over `prompt_batch_size` so readers can see how the speedup grows.

**Configuration:**

- Model: `Qwen/Qwen2.5-0.5B-Instruct` (the §11.7 trainee — same as the comparative-trainer notebook)
- Max completion length: 512 tokens
- Sampling: `temperature=0.7, top_p=0.9` (typical rollout settings)
- Sweep: `prompt_batch_size ∈ (1, 8, 32, 128)` × `num_generations=4` per prompt

**Estimated runtime:** ~20 min on Colab A100-40GB. **Cost:** ~$0.50.

**Open in Colab:** [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stateset/stateset-agents/blob/master/notebooks/vllm_speedup_benchmark.ipynb)

The artifact this notebook produces is `benchmark_results/whitepaper_v1/vllm_speedup_qwen25_05b_instruct.json`, referenced from whitepaper §6.4. Reproduce the headline number for any (model, hardware, batch) by editing cell 5 and re-running.


## 1. Pin + install

In [ ]:
import os, subprocess
PINNED_COMMIT = '8fc11b4'
if not os.path.exists('/content/stateset-agents'):
    subprocess.check_call(['git', 'clone', '--quiet',
        'https://github.com/stateset/stateset-agents', '/content/stateset-agents'])
subprocess.check_call(['git', '-C', '/content/stateset-agents', 'checkout', '--quiet', PINNED_COMMIT])
%cd /content/stateset-agents
print('Pinned to', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD']).decode().strip())

In [ ]:
# vLLM is the key dependency here — install via the [vllm] extra. Cell ~3 min on Colab.
%pip install --quiet -e '.[training,vllm]'
%pip install --quiet -U transformers accelerate
print('Install complete. Runtime > Restart session, then re-run from cell 1.')

## 2. Setup

In [ ]:
MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
MAX_TOKENS = 512
NUM_GENERATIONS = 4
TEMPERATURE = 0.7
TOP_P = 0.9
BATCH_SIZES = [1, 8, 32, 128]   # prompt_batch_size sweep
SAMPLE_PROMPTS = [
    'Solve this step by step. If a train travels 60 mph for 2.5 hours, how far does it go?',
    'Write a polite refund acknowledgment for order #1234.',
    'Explain the difference between TCP and UDP in two sentences.',
    'Suggest three exercises for someone starting strength training.',
    'List the pros and cons of using Rust for systems programming.',
    'Help me debug this Python error: TypeError: unsupported operand type(s).',
    'What is the capital of France, and what river runs through it?',
    'Recommend a beginner-friendly book about machine learning.',
] * 16   # 128 prompts, recycled — enough for the largest batch

import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

## 3. HF baseline generate

In [ ]:
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f'Loading HF model: {MODEL}')
hf_tokenizer = AutoTokenizer.from_pretrained(MODEL)
hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.bfloat16, device_map='cuda', attn_implementation='sdpa',
)
hf_model.eval()

@torch.no_grad()
def hf_generate_batch(prompts, num_generations):
    """Generate `num_generations` completions per prompt using HF model.generate."""
    # Replicate each prompt num_generations times — same shape vLLM produces.
    expanded = []
    for p in prompts:
        expanded.extend([p] * num_generations)
    inputs = hf_tokenizer(expanded, return_tensors='pt', padding=True, truncation=True, max_length=256).to('cuda')
    outputs = hf_model.generate(
        **inputs,
        max_new_tokens=MAX_TOKENS,
        temperature=TEMPERATURE, top_p=TOP_P, do_sample=True,
        pad_token_id=hf_tokenizer.eos_token_id,
    )
    return hf_tokenizer.batch_decode(outputs, skip_special_tokens=True)

# Warm up
print('Warming up HF...')
hf_generate_batch(SAMPLE_PROMPTS[:2], 1)
print('Warm.')

## 4. vLLM

In [ ]:
from vllm import LLM, SamplingParams

print(f'Loading vLLM: {MODEL}')
vllm_llm = LLM(
    model=MODEL,
    dtype='bfloat16',
    gpu_memory_utilization=0.5,   # leave room for the HF model still in memory
    max_model_len=1024,
    enable_prefix_caching=True,
)
sampling = SamplingParams(
    n=NUM_GENERATIONS,           # vLLM generates n completions per prompt natively
    max_tokens=MAX_TOKENS, temperature=TEMPERATURE, top_p=TOP_P,
)
print('vLLM ready.')

## 5. Sweep + measure

In [ ]:
import json
import gc

results = []
for batch_size in BATCH_SIZES:
    prompts = SAMPLE_PROMPTS[:batch_size]

    # HF generate timing — N=3 iters per batch size, take median to suppress warmup noise
    hf_times = []
    for _ in range(3):
        t0 = time.perf_counter()
        _ = hf_generate_batch(prompts, NUM_GENERATIONS)
        hf_times.append(time.perf_counter() - t0)
    hf_median = sorted(hf_times)[1]
    hf_throughput_tok_s = (batch_size * NUM_GENERATIONS * MAX_TOKENS) / hf_median

    # vLLM timing — also N=3
    vllm_times = []
    for _ in range(3):
        t0 = time.perf_counter()
        _ = vllm_llm.generate(prompts, sampling, use_tqdm=False)
        vllm_times.append(time.perf_counter() - t0)
    vllm_median = sorted(vllm_times)[1]
    vllm_throughput_tok_s = (batch_size * NUM_GENERATIONS * MAX_TOKENS) / vllm_median

    speedup = hf_median / vllm_median
    row = {
        'prompt_batch_size': batch_size,
        'num_generations': NUM_GENERATIONS,
        'max_tokens': MAX_TOKENS,
        'effective_batch': batch_size * NUM_GENERATIONS,
        'hf_seconds': round(hf_median, 3),
        'vllm_seconds': round(vllm_median, 3),
        'hf_throughput_tok_s': round(hf_throughput_tok_s, 0),
        'vllm_throughput_tok_s': round(vllm_throughput_tok_s, 0),
        'speedup_ratio': round(speedup, 2),
    }
    print(f"batch={batch_size:4d} (eff. {row['effective_batch']:5d}): "
          f"HF {hf_median:7.2f}s vs vLLM {vllm_median:7.2f}s → {speedup:6.2f}×  "
          f"({int(vllm_throughput_tok_s)} vs {int(hf_throughput_tok_s)} tok/s)")
    results.append(row)
    gc.collect(); torch.cuda.empty_cache()

## 6. Save schema-compliant result

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

result = {
    'benchmark': 'vllm_vs_hf_generate',
    'model': MODEL,
    'commit': PINNED_COMMIT,
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'config': {
        'num_generations': NUM_GENERATIONS,
        'max_tokens': MAX_TOKENS,
        'temperature': TEMPERATURE,
        'top_p': TOP_P,
    },
    'sweep': results,
    'hardware': {
        'gpu': torch.cuda.get_device_name(0),
        'cuda': torch.version.cuda,
        'peak_vram_mb': torch.cuda.max_memory_allocated() // (1024**2),
    },
    'notes': 'HF generate baseline uses sdpa attention, bfloat16. vLLM uses paged-attention. Both use the same prompts, same sampling params (temperature, top_p), same max_tokens. The speedup_ratio is hf_seconds / vllm_seconds — values > 1 mean vLLM is faster. Effective batch = prompt_batch_size × num_generations.',
}

out = Path('/content/vllm_speedup_qwen25_05b.json')
out.write_text(json.dumps(result, indent=2))
print(json.dumps(result, indent=2))
print(f'\nSaved to: {out}')

print('\n--- Headline number for §6.4 ---')
target_row = [r for r in results if r['prompt_batch_size'] == 32]
if target_row:
    r = target_row[0]
    print(f"At prompt_batch_size=32, num_generations=4 (effective 128), max_tokens=512:")
    print(f"  HF: {r['hf_seconds']}s ({r['hf_throughput_tok_s']:.0f} tok/s)")
    print(f"  vLLM: {r['vllm_seconds']}s ({r['vllm_throughput_tok_s']:.0f} tok/s)")
    print(f"  Speedup: {r['speedup_ratio']}×")

## 7. What this fills in for the whitepaper

§6.4 of the whitepaper currently says vLLM is "3× to 20×" faster than HF generate depending on configuration. This notebook produces the first-party number for one specific configuration (the one §6.4 names as the validation target) plus a sweep showing how the speedup grows with batch size.

Save the result JSON to `benchmark_results/whitepaper_v1/vllm_speedup_qwen25_05b_instruct.json` and cite the relevant row from §6.4's "Validated configuration" paragraph.
